# 🏥 Hybrid Clinical Decision Support System - Colab Training

This notebook trains a multi-class skin disease classifier using:
- **Melanoma** Detection
- **Eczema** Detection
- **Psoriasis** Detection  
- **Acne** Detection
- **Normal/Healthy Skin** Detection

**Updated Features:**
- Uses 5 individual Kaggle datasets with pre-split train/val/test folders
- Higher resolution (512x512) for better detail preservation
- GPU acceleration for faster training (TPU also supported)
- Automatically detects and uses GPU/TPU/CPU
- Organized structure: each disease has its own folder with proper splits

**Datasets:**
- `mateenzahid/acne-dataset`
- `mateenzahid/eczema-dataset`
- `mateenzahid/melanoma-dataset`
- `mateenzahid/normal-skin-dataset`
- `mateenzahid/psoriasis-dataset`

**Hardware Requirements:**
- GPU: Recommended (Runtime → Change runtime type → GPU - T4, V100, or A100)
- TPU: Also supported (Runtime → Change runtime type → TPU)
- RAM: 12.7 GB High-RAM runtime recommended for 512x512 images
- Disk: ~2-3GB for all 5 datasets and models

## 📦 Step 1: Setup Environment

In [9]:
# Check hardware availability (GPU/TPU)
import torch
import os

print(f"PyTorch version: {torch.__version__}")

# Check for GPU first (prioritized)
if torch.cuda.is_available():
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    device = torch.device('cuda')
    use_tpu = False
else:
    # Fall back to TPU if GPU not available
    try:
        import torch_xla
        device = torch_xla.device()
        print(f"✓ TPU detected: {device}")
        print(f"TPU cores available")
        use_tpu = True
    except:
        use_tpu = False
        print("⚠️ No GPU/TPU detected. Training will be slower on CPU.")
        print("Consider: Runtime → Change runtime type → Hardware accelerator → GPU")
        device = torch.device('cpu')

print(f"\nUsing device: {device}")

PyTorch version: 2.10.0+cu128
✓ GPU detected: Tesla T4
GPU Memory: 14.6 GB

Using device: cuda


In [10]:
# Install required packages
!pip install -q timm pyyaml gradio kaggle requests tqdm scikit-learn pillow opencv-python shap

# Install TPU support if using TPU (optional, GPU is prioritized)
try:
    import torch_xla
    print("✓ TPU libraries already installed")
except:
    print("TPU libraries not installed (GPU will be used if available)")
    # Uncomment below if you want to install TPU support:
    # !pip install -q cloud-tpu-client==0.10 torch-xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

TPU libraries not installed (GPU will be used if available)


## 💾 Step 2: Mount Google Drive (Optional - for saving models)

In [11]:
# Mount Google Drive to save trained models
from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Drive
import os
project_dir = '/content/drive/MyDrive/clinical_diagnosis_system'
os.makedirs(project_dir, exist_ok=True)
print(f"✓ Project directory: {project_dir}")

Mounted at /content/drive
✓ Project directory: /content/drive/MyDrive/clinical_diagnosis_system


## 📥 Step 3: Clone Project from GitHub (or upload files)

In [12]:
# Option A: Clone from GitHub (if you have a repo)
# !git clone https://github.com/yourusername/your-repo.git
# %cd your-repo

# Option B: Upload project files manually
# Use Colab's file upload feature to upload your project zip

# Option C: Start fresh (download individual files)
# This notebook assumes you have the project structure

# For this example, we'll create minimal structure
import os
os.makedirs('src', exist_ok=True)
os.makedirs('config', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

print("✓ Directory structure created")

✓ Directory structure created


## 🔑 Step 4: Setup Kaggle API Credentials

In [13]:
# Upload your kaggle.json file
# Download from: https://www.kaggle.com/settings → API → Create New API Token

from google.colab import files
import os

print("📤 Upload your kaggle.json file when prompted...")
uploaded = files.upload()

# Setup Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✓ Kaggle API configured!")

📤 Upload your kaggle.json file when prompted...


Saving kaggle.json to kaggle (1).json
✓ Kaggle API configured!


## 📊 Step 5: Download Datasets

In [ ]:
# Download individual skin disease datasets from Kaggle
import kaggle
from pathlib import Path

data_dir = Path('data')
raw_dir = data_dir / 'skin_lesions_raw'
raw_dir.mkdir(parents=True, exist_ok=True)

# Define datasets to download (5 separate datasets with train/val/test splits)
DATASETS = {
    'acne': 'mateenzahid/acne-dataset',
    'eczema': 'mateenzahid/eczema-dataset',
    'melanoma': 'mateenzahid/melanoma-dataset',
    'normal': 'mateenzahid/normal-skin-dataset',
    'psoriasis': 'mateenzahid/psoriasis-dataset'
}

print("📥 Downloading individual skin disease datasets...")
print("This will download 5 separate datasets:\n")

for disease, dataset_id in DATASETS.items():
    print(f"📦 Downloading {disease.upper()} dataset...")
    print(f"   Dataset: {dataset_id}")
    
    disease_dir = raw_dir / disease
    disease_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        kaggle.api.dataset_download_files(
            dataset_id,
            path=str(disease_dir),
            unzip=True,
            quiet=False
        )
        print(f"   ✓ {disease.upper()} dataset downloaded!\n")
    except Exception as e:
        print(f"   ✗ Error downloading {disease} dataset: {e}\n")
        print("   Please check your Kaggle API credentials and dataset permissions\n")

print("\n✓ All datasets downloaded!")

# List downloaded structure
print("\n📁 Downloaded structure:")
for disease in DATASETS.keys():
    disease_path = raw_dir / disease
    if disease_path.exists():
        for split in ['train', 'val', 'test']:
            split_path = disease_path / split
            if split_path.exists():
                img_count = len(list(split_path.rglob('*.jpg'))) + len(list(split_path.rglob('*.png')))
                print(f"  - {disease}/{split}/  ({img_count} images)")


📥 Downloading unified skin disease dataset...
Dataset: mateenzahid/skin-diesease
This contains melanoma, eczema, psoriasis, acne, and normal skin images
Total size: ~819MB

Dataset URL: https://www.kaggle.com/datasets/mateenzahid/skin-diesease


100%|██████████| 748M/748M [00:04<00:00, 168MB/s]




✓ Dataset downloaded successfully!

📁 Downloaded structure:
  - skin_lesions_raw/  (24298 images)


## 🔧 Step 6: Verify Dataset Structure

The unified dataset already contains organized train/val/test splits, so we'll use it directly without reorganization.

In [ ]:
# Verify dataset structure with train/val/test splits
from pathlib import Path

# Dataset root is the raw directory containing disease folders
data_root = Path('data/skin_lesions_raw')

if not data_root.exists():
    print(f'⚠️ Dataset root not found: {data_root}')
    print('Please run the download cell first.')
else:
    print(f'Using dataset root: {data_root}\n')
    
    # Expected structure: data_root/disease_name/split/images
    diseases = ['acne/Acne', 'eczema/Eczema', 'melanoma/melanoma', 'normal/Normal', 'psoriasis/PSORIASIS']
    splits = ['train', 'val', 'test']
    
    print('📊 Dataset Summary:\n')
    total_images = 0
    
    for disease in diseases:
        disease_path = data_root / disease
        if not disease_path.exists():
            print(f"⚠️ Missing: {disease}/")
            continue
            
        print(f"{disease.upper()}:")
        disease_total = 0
        
        for split in splits:
            split_path = disease_path / split
            if split_path.exists():
                count = 0
                for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
                    count += len(list(split_path.rglob(ext)))
                print(f"  - {split}: {count} images")
                disease_total += count
                total_images += count
            else:
                print(f"  - {split}: ⚠️ missing")
        
        print(f"  Total: {disease_total}\n")
    
    print(f"📈 Grand Total: {total_images} images")
    print('\n✓ Dataset structure verified!')


📁 Verifying dataset structure...

✓ melanoma:
  - Total images: 10605
✓ eczema:
  - Total images: 3123
✓ psoriasis:
  - Total images: 2801
✓ acne:
  - train/: 2778 images
  - test/: 918 images
  - valid/: 921 images
✓ normal:
  - Total images: 3152

✓ Dataset structure verified!
We'll use the existing folder structure directly (no copying/reorganization needed)


## 🎓 Step 7: Train the CNN Model

In [ ]:
# Load data from individual disease folders with train/val/test splits
from pathlib import Path

def prepare_data():
    """Load data from disease-specific folders with splits: train/val/test.
    Returns: train_paths, train_labels, val_paths, val_labels, test_paths, test_labels, class_names
    """
    root = Path('data/skin_lesions_raw')
    
    # Class names (full descriptive names, matching order of folder names)
    class_names = [
        "Acne and Rosacea Photos",
        "Eczema Photos",
        "Melanoma Skin Cancer Nevi and Moles",
        "Normal Healthy Skin",
        "Psoriasis pictures Lichen Planus and related diseases"
    ]
    
    # Directory short names (must match folders)
    short_names = ['acne/Acne', 'eczema/Eczema', 'melanoma/melanoma', 'normal/Normal', 'psoriasis/PSORIASIS']
    class_to_idx = {full: i for i, full in enumerate(class_names)}
    short_to_full = dict(zip(short_names, class_names))
    
    train_paths, train_labels = [], []
    val_paths, val_labels = [], []
    test_paths, test_labels = [], []
    
    for short in short_names:
        full = short_to_full[short]
        idx = class_to_idx[full]
        
        disease_path = root / short
        
        for split, paths_list, labels_list in (
            ('train', train_paths, train_labels),
            ('val', val_paths, val_labels),
            ('test', test_paths, test_labels),
        ):
            split_path = disease_path / split
            
            if not split_path.exists():
                print(f"⚠️ Missing folder: {split_path}")
                continue
            
            # Collect all image files from this split
            for img_path in sorted(split_path.rglob('*')):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                    paths_list.append(str(img_path))
                    labels_list.append(idx)
    
    # Print summary
    print("\n📊 Data Loading Summary:\n")
    for split_name, paths, labels in (
        ('Train', train_paths, train_labels), 
        ('Validation', val_paths, val_labels), 
        ('Test', test_paths, test_labels)
    ):
        counts = {name: 0 for name in class_names}
        for lab in labels:
            counts[class_names[lab]] += 1
        
        print(f"{split_name} Set:")
        total = 0
        for name in class_names:
            if counts[name] > 0:
                print(f"  - {name}: {counts[name]}")
                total += counts[name]
        print(f"  Total: {total}\n")
    
    return train_paths, train_labels, val_paths, val_labels, test_paths, test_labels, class_names

# Execute to load data
train_paths, train_labels, val_paths, val_labels, test_paths, test_labels, class_names = prepare_data()


📊 Loading images from dataset...

✓ melanoma: 10605 images total → 8484 train, 2121 val
✓ eczema: 3123 images total → 2498 train, 625 val
✓ psoriasis: 2806 images total → 2244 train, 562 val
✓ acne/train: 2778 images
✓ acne/val: 921 images
✓ normal: 3152 images total → 2521 train, 631 val

📈 Dataset Summary:
Training samples: 18525
Validation samples: 4860
Total images: 23385


In [ ]:
# Minimal dataset class for this notebook (defines SkinLesionDataset)
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

class SkinLesionDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img_path = self.paths[idx]
        label = int(self.labels[idx])
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            # If an image fails to load, return a black image of the expected size
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0,0,0))
        if self.transform is not None:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

In [ ]:
# Define transforms that preserve original image sizes (adaptive approach)
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch

class SkinLesionDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, label

# Find a reasonable common size based on the dataset
# We'll use a large size that preserves detail but is GPU-friendly
# 512x512 is a good balance between original quality and memory usage
IMG_SIZE = 512

# Use minimal resizing to preserve original image quality
# We resize to a common dimension to enable batching
common_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),  # Resize to common size for batching
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Same transform for all splits
train_transform = common_transform
val_transform = common_transform
test_transform = common_transform

# Create datasets and dataloaders
train_dataset = SkinLesionDataset(train_paths, train_labels, train_transform)
val_dataset = SkinLesionDataset(val_paths, val_labels, val_transform)
test_dataset = SkinLesionDataset(test_paths, test_labels, test_transform)

# Adjust batch size based on IMG_SIZE and GPU memory
# Larger images require smaller batch sizes
batch_size = 8 if IMG_SIZE >= 512 else 16
num_workers = 4

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"✓ Data Loading Configuration:")
print(f"  - Image size: {IMG_SIZE}x{IMG_SIZE} (preserving more detail)")
print(f"  - Batch size: {batch_size}")
print(f"  - Num workers: {num_workers}")
print(f"\n📊 Dataset Sizes:")
print(f"  - Train samples: {len(train_dataset)}")
print(f"  - Val samples: {len(val_dataset)}")
print(f"  - Test samples: {len(test_dataset)}")
print(f"  - Batches per epoch: {len(train_loader)}")


Image size: 384x384
Batch size: 16
Batches per epoch: 1158


In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm

# Device setup
if torch.cuda.is_available():
    device = torch.device('cuda')
    use_tpu = False
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    try:
        import torch_xla
        device = torch_xla.device()
        use_tpu = True
        print(f"✓ Using TPU: {device}")
    except:
        device = torch.device('cpu')
        use_tpu = False
        print(f"⚠️ Using CPU (slower)")

# Model
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5)
model = model.to(device)

# Loss
criterion = nn.CrossEntropyLoss()

# Debug
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("✓ Model initialized")

✓ Using GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model parameters: 4,013,953
✓ Model initialized


In [ ]:
# Training loop with TPU/GPU support
from tqdm import tqdm
def train_model(model, train_loader, val_loader, epochs=10):
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    # Check if using TPU
    try:
        import torch_xla
        import torch_xla.core.xla_model as xm
        use_tpu = True
    except:
        use_tpu = False

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()

            if use_tpu:
                xm.optimizer_step(optimizer)  # TPU-specific optimizer step
            else:
                optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            pbar.set_postfix({'loss': train_loss/len(pbar), 'acc': 100.*train_correct/train_total})

        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = 100. * val_correct / val_total

        print(f"\nEpoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc

            # For TPU, move model to CPU before saving
            if use_tpu:
                cpu_model = model.cpu()
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': cpu_model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc,
                    'class_names': class_names
                }, 'models/cnn_skin_lesion.pth')
                model = model.to(device)  # Move back to TPU
            else:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc,
                    'class_names': class_names
                }, 'models/cnn_skin_lesion.pth')

            print(f"  ✓ Best model saved (Val Acc: {val_acc:.2f}%)")

    return history

# Train for 10 epochs (increase for better results)
history = train_model(model, train_loader, val_loader, epochs=10)

Epoch 1/10: 100%|██████████| 1158/1158 [05:10<00:00,  3.73it/s, loss=0.293, acc=88.1]



Epoch 1:
  Train Loss: 0.2925, Train Acc: 88.15%
  Val Loss: 0.1654, Val Acc: 92.35%
  ✓ Best model saved (Val Acc: 92.35%)


Epoch 2/10: 100%|██████████| 1158/1158 [05:11<00:00,  3.72it/s, loss=0.29, acc=87.8]



Epoch 2:
  Train Loss: 0.2900, Train Acc: 87.82%
  Val Loss: 0.1471, Val Acc: 91.58%


Epoch 3/10:  33%|███▎      | 387/1158 [01:44<03:10,  4.04it/s, loss=0.0637, acc=91.2]

## 📊 Step 8: Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(history['train_acc'], label='Train Accuracy')
ax2.plot(history['val_acc'], label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"\n🎯 Final Results:")
print(f"  Best Validation Accuracy: {max(history['val_acc']):.2f}%")
print(f"  Final Training Accuracy: {history['train_acc'][-1]:.2f}%")

## 💾 Step 9: Save Model to Google Drive

In [ ]:
# Copy trained model to Google Drive
!cp models/cnn_skin_lesion.pth /content/drive/MyDrive/clinical_diagnosis_system/

print("✓ Model saved to Google Drive!")
print("You can now download it and use it locally.")

## 🧪 Step 10: Test the Model

In [ ]:
# Full evaluation on `data/test/` — classification report + confusion matrix
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

# Convert to numpy
y_true = np.array(all_labels)
y_pred = np.array(all_preds)

# Class names (full)
labels = class_names

# Classification report
report = classification_report(y_true, y_pred, target_names=labels, digits=4, output_dict=True)
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=labels, digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Save report and confusion matrix numbers
out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)
with open(out_dir / 'test_classification_report.json', 'w') as fh:
    json.dump(report, fh, indent=2)
np.save(out_dir / 'test_confusion_matrix.npy', cm)
print(f"Saved report to {out_dir / 'test_classification_report.json'} and confusion matrix to {out_dir / 'test_confusion_matrix.npy'}")


## ✅ Summary

You've successfully:
1. ✓ Set up the environment with GPU support (TPU optional)
2. ✓ Downloaded 5 individual skin disease datasets from Kaggle
3. ✓ Loaded images with preserved quality (512x512 vs 384x384)
4. ✓ Trained an EfficientNet-B0 model on 5 disease classes
5. ✓ Visualized training results
6. ✓ Saved the model to Google Drive

**Dataset Info:**
- **Acne Dataset**: https://www.kaggle.com/datasets/mateenzahid/acne-dataset
- **Eczema Dataset**: https://www.kaggle.com/datasets/mateenzahid/eczema-dataset
- **Melanoma Dataset**: https://www.kaggle.com/datasets/mateenzahid/melanoma-dataset
- **Normal Skin Dataset**: https://www.kaggle.com/datasets/mateenzahid/normal-skin-dataset
- **Psoriasis Dataset**: https://www.kaggle.com/datasets/mateenzahid/psoriasis-dataset

**Structure:**
```
data/skin_lesions_raw/
├── acne/
│   ├── train/
│   ├── val/
│   └── test/
├── eczema/
│   ├── train/
│   ├── val/
│   └── test/
├── melanoma/
│   ├── train/
│   ├── val/
│   └── test/
├── normal/
│   ├── train/
│   ├── val/
│   └── test/
└── psoriasis/
    ├── train/
    ├── val/
    └── test/
```

**Model Features:**
- Higher resolution: 512x512 (better detail preservation)
- Pre-split datasets (no manual splitting needed)
- GPU-accelerated training
- EfficientNet-B0 backbone with transfer learning

### Next Steps:
- Download the model from Google Drive (`cnn_skin_lesion.pth`)
- Place it in your local project: `models/cnn_skin_lesion.pth`
- Run the web interface: `python app.py`
- Test with both diseased and healthy skin images

### Tips for Better Results:
- Train for 20-50 epochs (this notebook uses 10 for speed)
- Use learning rate scheduling
- Add data augmentation for training set
- Try ensemble methods with multiple models
- Consider mixed precision training (AMP) for faster GPU training
- Fine-tune batch size based on GPU memory (8-16 works well for 512x512 images)

In [ ]:
# --- Evaluation & Verification Utilities
# Evaluate trained CNN on per-disease test splits, verify single images,
# and optionally compare with existing saved report files (e.g., in ~/Downloads).

from pathlib import Path
import torch
import timm
import json
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

# Robust references to notebook variables if available
try:
    IMG_SIZE
    test_transform
    class_names
except NameError:
    IMG_SIZE = 512
    from torchvision import transforms
    test_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    class_names = [
        "Acne and Rosacea Photos",
        "Eczema Photos",
        "Melanoma Skin Cancer Nevi and Moles",
        "Normal Healthy Skin",
        "Psoriasis pictures Lichen Planus and related diseases"
    ]

# Load model helper
def load_cnn_model(model_path='models/cnn_skin_lesion.pth', device=None):
    device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
    model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=len(class_names))
    if not Path(model_path).exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")
    ckpt = torch.load(model_path, map_location=device)
    # checkpoint might contain 'model_state_dict' or be raw state_dict
    state = ckpt.get('model_state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state)
    model.to(device).eval()
    return model, device

# Evaluate lists of paths/labels
def evaluate_paths(paths, labels, model, device, batch_size=16, num_workers=4):
    ds = SkinLesionDataset(paths, labels, transform=test_transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labs in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labs.numpy().tolist())
    report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, digits=4)
    cm = confusion_matrix(all_labels, all_preds)
    return report, cm

# Evaluate entire test set assembled from per-disease /test/ folders
def evaluate_test_splits(data_root='data/skin_lesions_raw', model_path='models/cnn_skin_lesion.pth', save_dir='outputs', batch_size=16):
    root = Path(data_root)
    if not root.exists():
        raise FileNotFoundError(f"Data root not found: {root}")

    diseases = ['acne', 'eczema', 'melanoma', 'normal', 'psoriasis']
    short_to_full = {
        'acne': class_names[0],
        'eczema': class_names[1],
        'melanoma': class_names[2],
        'normal': class_names[3],
        'psoriasis': class_names[4]
    }

    test_paths, test_labels = [], []
    for idx, short in enumerate(diseases):
        split_path = root / short / 'test'
        if not split_path.exists():
            print(f"⚠️ Missing test folder for {short}: {split_path}")
            continue
        for p in sorted(split_path.rglob('*')):
            if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                test_paths.append(str(p))
                test_labels.append(idx)

    print(f"Found {len(test_paths)} test images across diseases")
    model, device = load_cnn_model(model_path)
    report, cm = evaluate_paths(test_paths, test_labels, model, device, batch_size=batch_size)

    out = Path(save_dir)
    out.mkdir(parents=True, exist_ok=True)
    rpt_path = out / 'test_classification_report_from_notebook.json'
    cm_path = out / 'test_confusion_matrix_from_notebook.npy'

    with open(rpt_path, 'w') as fh:
        json.dump(report, fh, indent=2)
    np.save(cm_path, cm)

    print(f"Saved classification report: {rpt_path}")
    print(f"Saved confusion matrix: {cm_path}")
    return report, cm

# Single image verification
def predict_single_image(image_path, model_path='models/cnn_skin_lesion.pth', topk=5):
    from PIL import Image
    model, device = load_cnn_model(model_path)
    img = Image.open(image_path).convert('RGB')
    inp = test_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(inp)
        probs = torch.softmax(out, dim=1).cpu().numpy()[0]
    probs_dict = {class_names[i]: float(probs[i]) for i in range(len(class_names))}
    sorted_probs = sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)[:topk]
    return sorted_probs, probs_dict

# Compare notebook-generated report with an existing JSON report (e.g., from Downloads)
def compare_reports(existing_report_path, new_report_dict):
    existing = None
    try:
        with open(existing_report_path, 'r') as fh:
            existing = json.load(fh)
    except Exception as e:
        print(f"Could not load existing report: {e}")
        return

    # Compare per-class f1-score if available
    print("Per-class F1 comparison:")
    for cls in class_names:
        e_f1 = existing.get(cls, {}).get('f1-score') if isinstance(existing, dict) else None
        n_f1 = new_report_dict.get(cls, {}).get('f1-score') if isinstance(new_report_dict, dict) else None
        print(f" - {cls}: existing f1={e_f1}, new f1={n_f1}")

# Usage examples:
# report, cm = evaluate_test_splits(data_root='data/skin_lesions_raw', model_path='models/cnn_skin_lesion.pth')
# preds, probs = predict_single_image('/path/to/my_image.jpg')
# compare_reports('/home/dev/Downloads/test_classification_report.json', report)

print('Evaluation & verification utilities loaded.\nExamples:')
print(" - evaluate_test_splits()  # runs full test evaluation and saves outputs to `outputs/`")
print(" - predict_single_image('/path/to/image.jpg')  # returns top predictions and full prob dict")
print(" - compare_reports('/path/to/existing_report.json', report)  # compare reports")


In [ ]:
# --- Single-image upload widget (Jupyter-safe)
# Upload an image and run prediction with the loaded model
import torch
import timm
import numpy as np
from PIL import Image
from io import BytesIO
from torchvision import transforms
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Config
IMG_SIZE = 512
class_names = [
    "Acne and Rosacea Photos",
    "Eczema Photos",
    "Melanoma Skin Cancer Nevi and Moles",
    "Normal Healthy Skin",
    "Psoriasis pictures Lichen Planus and related diseases"
]

# Transform (matches model input)
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load model helper (re-uses model saved in `models/`)
def load_model(model_path='models/cnn_skin_lesion.pth'):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=len(class_names))
    if not Path(model_path).exists():
        raise FileNotFoundError(f"Model not found: {model_path}")
    ckpt = torch.load(model_path, map_location=device)
    state = ckpt.get('model_state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state)
    model.to(device).eval()
    print(f"✅ Model loaded on {device}")
    return model, device

# Try to load model once (fail gracefully)
try:
    MODEL, DEVICE = load_model()
except Exception as e:
    MODEL, DEVICE = None, None
    print(f"Model load warning: {e}")

# Prediction function
def predict_image(img):
    if MODEL is None:
        raise RuntimeError("Model not loaded. Run load_model() or place `models/cnn_skin_lesion.pth`.")
    inp = test_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = MODEL(inp)
        probs = torch.softmax(out, dim=1).cpu().numpy()[0]
    results = sorted([(class_names[i], float(probs[i])) for i in range(len(class_names))], key=lambda x: x[1], reverse=True)
    return results

# Upload widget
uploader = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False)
display(uploader)

# Handler
def on_upload_change(change):
    if not uploader.value:
        return
    for name, file_info in uploader.value.items():
        try:
            img_bytes = file_info['content']
            img = Image.open(BytesIO(img_bytes)).convert('RGB')
        except Exception as e:
            print(f"Error loading uploaded file: {e}")
            return

        try:
            preds = predict_image(img)
        except Exception as e:
            print(f"Prediction error: {e}")
            return

        print("\n🔍 Top Predictions:")
        for cls, p in preds[:5]:
            print(f"{cls}: {p:.4f}")

uploader.observe(on_upload_change, names='value')

print('\nUpload an image above to run prediction.')
